In [13]:
# 1) GPU 가 사용 가능한지 확인
import torch

print("PyTorch 버전:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM(GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    raise RuntimeError("GPU가 없습니다. [런타임 → 런타임 유형 변경 → T4 GPU] 후 다시 실행하세요.")


PyTorch 버전: 2.11.0+cu128
CUDA 사용 가능: True
GPU: Tesla T4
VRAM(GB): 15.64


In [5]:
# 2) ComfyUI 본체 + GGUF 노드(플러그인) 설치
import os

%cd /content
if not os.path.isdir("/content/ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git   # ComfyUI 본체

%cd /content/ComfyUI
!pip install -q -r requirements.txt                            # ComfyUI 의존성

# GGUF 파일을 읽기 위한 커스텀 노드 설치
%cd /content/ComfyUI/custom_nodes
if not os.path.isdir("ComfyUI-GGUF"):
    !git clone https://github.com/city96/ComfyUI-GGUF.git      # GGUF 로더 노드
%cd /content/ComfyUI
!pip install -q gguf                                           # GGUF 파싱 라이브러리

print("\n완료: ComfyUI + GGUF 노드 설치")

/content
Cloning into 'ComfyUI'...
remote: Enumerating objects: 43864, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 43864 (delta 91), reused 65 (delta 62), pack-reused 43728 (from 2)
Receiving objects: 100% (43864/43864), 84.50 MiB | 19.57 MiB/s, done.
Resolving deltas: 100% (29754/29754), done.
/content/ComfyUI
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.7/20.7 MB 92.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.8/346.8 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 111.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.1/22.1 MB 91.

In [7]:
# 3) 파이프라인 구성 파일 3종 다운로드 → ComfyUI 표준 폴더에 배치
import os
%cd /content/ComfyUI

# ComfyUI 가 모델을 찾는 표준 경로 생성
os.makedirs("models/diffusion_models", exist_ok=True)   # 디퓨전 모델
os.makedirs("models/text_encoders",  exist_ok=True)   # 텍스트 인코더
os.makedirs("models/vae",            exist_ok=True)   # VAE

def 다운로드(url, 저장경로):
    """이미 받았으면 건너뛰고, 아니면 wget 로 받는 헬퍼"""
    if os.path.exists(저장경로) and os.path.getsize(저장경로) > 1_000_000:
        print(f"  ↳ 이미 있음: {저장경로} ({os.path.getsize(저장경로)/1e9:.2f} GB)")
        return
    print(f"  ↳ 다운로드 중...")
    !wget -q --show-progress -O "{저장경로}" "{url}"

print("[1/3] 디퓨전 모델 (GGUF Q4, 약 2.6GB)")
다운로드("https://huggingface.co/unsloth/FLUX.2-klein-4B-GGUF/resolve/main/flux-2-klein-4b-Q4_K_M.gguf",
        "models/diffusion_models/flux-2-klein-4b-Q4_K_M.gguf")

print("[2/3] 텍스트 인코더 (Qwen3-4B, 약 8GB)")
다운로드("https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-4b/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors",
        "models/text_encoders/qwen_3_4b.safetensors")

print("[3/3] VAE (약 330MB)")
다운로드("https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-4b/resolve/main/split_files/vae/flux2-vae.safetensors",
        "models/vae/flux2-vae.safetensors")

print("\n다운로드 완료. 배치된 파일 확인:")
!ls -lh models/diffusion_models/flux-2-klein-4b-Q4_K_M.gguf \
      models/text_encoders/qwen_3_4b.safetensors \
      models/vae/flux2-vae.safetensors

/content/ComfyUI
[1/3] 디퓨전 모델 (GGUF Q4, 약 2.6GB)
  ↳ 다운로드 중...
models/diffusion_mo 100%[===================>]   2.42G  60.8MB/s    in 51s     
[2/3] 텍스트 인코더 (Qwen3-4B, 약 8GB)
  ↳ 다운로드 중...
models/text_encoder 100%[===================>]   7.49G   133MB/s    in 99s     
[3/3] VAE (약 330MB)
  ↳ 다운로드 중...
models/vae/flux2-va 100%[===================>] 320.64M   130MB/s    in 2.5s    

다운로드 완료. 배치된 파일 확인:
-rw-r--r-- 1 root root 2.5G Jul 31 13:13 models/diffusion_models/flux-2-klein-4b-Q4_K_M.gguf
-rw-r--r-- 1 root root 7.5G Jul 31 13:14 models/text_encoders/qwen_3_4b.safetensors
-rw-r--r-- 1 root root 321M Jul 31 13:15 models/vae/flux2-vae.safetensors


In [11]:
# 3.1) 저용량 ControlNet(OpenPose) 경로용 파일 설치 — SD1.5 기반, 총 4GB 미만
#     FLUX.2-klein 용 ControlNet은 8GB+ 라서 12GB RAM 한계를 넘기므로,
#     포즈 제어가 필요할 때만 훨씬 가벼운 SD1.5 + ControlNet 조합으로 전환합니다.
import os
%cd /content/ComfyUI

os.makedirs("models/checkpoints", exist_ok=True)
os.makedirs("models/controlnet", exist_ok=True)

print("[1/2] SD1.5 체크포인트 (fp16, 약 2.1GB)")
다운로드("https://huggingface.co/Comfy-Org/stable-diffusion-v1-5-archive/resolve/main/v1-5-pruned-emaonly-fp16.safetensors",
        "models/checkpoints/v1-5-pruned-emaonly-fp16.safetensors")

print("[2/2] ControlNet OpenPose (fp16, 약 1.4GB)")
다운로드("https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_openpose_fp16.safetensors",
        "models/controlnet/control_v11p_sd15_openpose_fp16.safetensors")

# 사진에서 자동으로 포즈 스켈레톤을 추출해주는 전처리 노드 설치
# (첫 실행 시 OpenPose 추정 모델(~200MB)을 자체적으로 내려받습니다)
%cd /content/ComfyUI/custom_nodes
if not os.path.isdir("comfyui_controlnet_aux"):
    !git clone https://github.com/Fannovel16/comfyui_controlnet_aux.git
%cd /content/ComfyUI/custom_nodes/comfyui_controlnet_aux
!pip install -q -r requirements.txt
%cd /content/ComfyUI

print("\n완료: 저용량 ControlNet 경로 준비 (SD1.5 체크포인트 + ControlNet + 포즈 추출 노드)")

/content/ComfyUI
[1/2] SD1.5 체크포인트 (fp16, 약 2.1GB)
  ↳ 다운로드 중...
models/checkpoints/ 100%[===================>]   1.99G   109MB/s    in 27s     
[2/2] ControlNet OpenPose (fp16, 약 1.4GB)
  ↳ 다운로드 중...
models/controlnet/c 100%[===================>] 689.13M  40.9MB/s    in 15s     
/content/ComfyUI/custom_nodes
Cloning into 'comfyui_controlnet_aux'...
remote: Enumerating objects: 5741, done.
remote: Counting objects: 100% (2029/2029), done.
remote: Compressing objects: 100% (447/447), done.
remote: Total 5741 (delta 1741), reused 1582 (delta 1582), pack-reused 3712 (from 4)
Receiving objects: 100% (5741/5741), 42.92 MiB | 16.87 MiB/s, done.
Resolving deltas: 100% (3128/3128), done.
/content/ComfyUI/custom_nodes/comfyui_controlnet_aux
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 k

### 3.2) 포즈 반영 방식 안내 (저용량 ControlNet, SD1.5 기반)

FLUX.2-klein-4B(GGUF)용 ControlNet은 8GB 이상의 추가 가중치가 필요해서, 이미 UNet(2.6GB)+텍스트 인코더(8GB)가 올라간 상태에서는 Colab 12GB RAM 한계를 넘깁니다. 그래서 **포즈를 지정할 때만** 훨씬 가벼운 SD1.5 + ControlNet OpenPose 조합(총 4GB 미만)으로 전환합니다.

- 일반 생성: 기존처럼 FLUX.2-klein-4B 사용 (고화질)
- 포즈 ControlNet 모드: SD1.5 체크포인트(2.1GB) + ControlNet OpenPose(1.4GB) + 포즈 추출 노드로 진행 (~4GB, 정확한 뼈대 고정 가능)
- 포즈 이미지는 스켈레톤이 아닌 **일반 사진을 업로드**해도 됩니다. 내부적으로 OpenPose 전처리 노드가 자동으로 뼈대를 추출합니다.
- SD1.5 특성상 512×512~512×768 해상도에서 가장 결과가 좋습니다.

In [15]:
# 4) ComfyUI 서버를 백그라운드로 실행 (Gradio 가 이 서버에 접속함)
#    ※ Gradio 를 쓰므로 외부 공개 터널은 필요 없습니다.
import subprocess, os, time

%cd /content/ComfyUI

# 혹시 떠 있는 예전 서버가 있으면 정리
!pkill -f "main.py" 2>/dev/null
time.sleep(2)

# ComfyUI 백그라운드 실행 (포트 8188)
os.system("nohup python main.py --listen 0.0.0.0 --port 8188 > /content/comfyui.log 2>&1 &")

# 서버가 뜰 때까지 대기 (로컬에서 200 응답이 오면 준비 완료)
print("ComfyUI 기동 대기 중...")
for _ in range(40):
    time.sleep(1)
    code = subprocess.run("curl -s -o /dev/null -w '%{http_code}' http://127.0.0.1:8188/",
                          shell=True, capture_output=True, text=True).stdout
    if code == "200":
        print("완료: ComfyUI 서버 준비 (http://127.0.0.1:8188)")
        break
else:
    print("서버가 안 뜹니다. 로그:")
    print(open("/content/comfyui.log").read()[-800:])

/content/ComfyUI
^C
ComfyUI 기동 대기 중...
완료: ComfyUI 서버 준비 (http://127.0.0.1:8188)


In [16]:
# 5) Gradio UI — 프롬프트를 입력하면 이미지를 생성하는 웹 화면
#    ComfyUI 의 API 를 호출해 백엔드로 사용합니다.
#    - 일반 생성: FLUX.2-klein-4B
#    - 포즈 ControlNet: SD1.5 + ControlNet OpenPose (저용량, ~4GB) — 12GB RAM 한계 대응
import gradio as gr
import requests, time, json, urllib.parse
import numpy as np
from PIL import Image as PILImage
import io

SERVER = "http://127.0.0.1:8188"   # Colab 안의 ComfyUI 서버 주소
MODE_POSE = "포즈 ControlNet (SD1.5, 저용량)"

# ---------- 핵심: ComfyUI API 로 이미지 생성하는 함수 ----------
def generate(prompt, negative, width, height, steps, seed, mode, pose_image, controlnet_strength):
    """프롬프트 → ComfyUI 워크플로우 JSON 조립 → API 호출 → 결과 이미지 반환"""

    is_pose_mode = (mode == MODE_POSE)

    if is_pose_mode:
        if pose_image is None:
            return None, "포즈 ControlNet 모드에서는 포즈 참고 이미지를 업로드해야 합니다."

        # 포즈 참고 이미지를 ComfyUI에 업로드 (스켈레톤이 아닌 일반 사진도 가능 — 자동으로 뼈대 추출됨)
        buffered = io.BytesIO()
        PILImage.fromarray(pose_image).save(buffered, format="PNG")
        try:
            files = {'image': ('pose_image.png', buffered.getvalue(), 'image/png')}
            r_upload = requests.post(f"{SERVER}/upload/image", files=files, timeout=30)
            if r_upload.status_code != 200:
                return None, f"포즈 이미지 업로드 실패 {r_upload.status_code}: {r_upload.text[:300]}"
            uploaded_filename = r_upload.json()['name']
        except Exception as e:
            return None, f"포즈 이미지 업로드 중 서버 연결 실패: {e}"

        # SD1.5 + ControlNet OpenPose 워크플로우 (총 모델 크기 ~4GB, 12GB RAM에 여유 있음)
        workflow = {
          "30": {"class_type": "CheckpointLoaderSimple", "inputs": {"ckpt_name": "v1-5-pruned-emaonly-fp16.safetensors"}},
          "31": {"class_type": "CLIPTextEncode", "inputs": {"text": prompt, "clip": ["30", 1]}},
          "32": {"class_type": "CLIPTextEncode", "inputs": {"text": negative or "", "clip": ["30", 1]}},
          "33": {"class_type": "EmptyLatentImage", "inputs": {"width": int(width), "height": int(height), "batch_size": 1}},
          "20": {"class_type": "LoadImage", "inputs": {"image": uploaded_filename}},
          "34": {"class_type": "OpenposePreprocessor", "inputs": {"image": ["20", 0], "detect_hand": "enable",
                                                                    "detect_body": "enable", "detect_face": "enable",
                                                                    "resolution": 512}},  # 사진 → 포즈 스켈레톤 자동 추출
          "21": {"class_type": "ControlNetLoader", "inputs": {"control_net_name": "control_v11p_sd15_openpose_fp16.safetensors"}},
          "22": {"class_type": "ControlNetApply", "inputs": {"conditioning": ["31", 0], "control_net": ["21", 0],
                                                               "image": ["34", 0], "strength": float(controlnet_strength)}},
          "23": {"class_type": "ControlNetApply", "inputs": {"conditioning": ["32", 0], "control_net": ["21", 0],
                                                               "image": ["34", 0], "strength": float(controlnet_strength)}},
          "35": {"class_type": "KSampler", "inputs": {"seed": int(seed), "steps": int(steps), "cfg": 7.0,
                                                        "sampler_name": "euler", "scheduler": "normal", "denoise": 1.0,
                                                        "model": ["30", 0], "positive": ["22", 0],
                                                        "negative": ["23", 0], "latent_image": ["33", 0]}},
          "36": {"class_type": "VAEDecode", "inputs": {"samples": ["35", 0], "vae": ["30", 2]}},
          "37": {"class_type": "SaveImage", "inputs": {"images": ["36", 0], "filename_prefix": "sd15_pose"}},
        }
        output_node = "37"
    else:
        # 일반 생성 워크플로우 (FLUX.2-klein-4B)
        workflow = {
          "10": {"class_type": "UnetLoaderGGUF",  "inputs": {"unet_name": "flux-2-klein-4b-Q4_K_M.gguf"}},   # 디퓨전 모델 로드
          "8":  {"class_type": "CLIPLoader",      "inputs": {"clip_name": "qwen_3_4b.safetensors", "type": "flux2"}},  # 텍스트 인코더 로드
          "6":  {"class_type": "CLIPTextEncode",  "inputs": {"text": prompt, "clip": ["8", 0]}},             # 프롬프트 인코딩
          "7":  {"class_type": "CLIPTextEncode",  "inputs": {"text": negative or "", "clip": ["8", 0]}},     # 네거티브 인코딩
          "5":  {"class_type": "EmptyLatentImage", "inputs": {"width": int(width), "height": int(height), "batch_size": 1}},  # 빈 캔버스
          "3":  {"class_type": "KSampler",        "inputs": {"seed": int(seed), "steps": int(steps), "cfg": 4.0,
                                                             "sampler_name": "euler", "scheduler": "simple", "denoise": 1.0,
                                                             "model": ["10", 0], "positive": ["6", 0],
                                                             "negative": ["7", 0], "latent_image": ["5", 0]}},  # 실제 생성
          "4":  {"class_type": "VAELoader",       "inputs": {"vae_name": "flux2-vae.safetensors"}},          # VAE 로드
          "9":  {"class_type": "VAEDecode",       "inputs": {"samples": ["3", 0], "vae": ["4", 0]}},         # latent → 이미지
          "11": {"class_type": "SaveImage",       "inputs": {"images": ["9", 0], "filename_prefix": "flux2_klein"}},  # 저장
        }
        output_node = "11"

    # ② 워크플로우를 ComfyUI 에 전송 (생성 요청)
    try:
        p = {"prompt": workflow, "client_id": "colab"}
        r = requests.post(f"{SERVER}/prompt", json=p, timeout=180) # Increased timeout
        if r.status_code != 200:
            return None, f"요청 실패 {r.status_code}: {r.text[:500]}"
        pid = r.json()["prompt_id"]   # 작업 ID
    except Exception as e:
        log_content = ""
        try:
            with open("/content/comfyui.log", "r") as f:
                log_content = f.read()
        except FileNotFoundError:
            log_content = "ComfyUI log file not found."

        error_message = f"서버 연결 실패 (4번 셀 실행 확인): {e}"
        if log_content:
            error_message += f"\nComfyUI 로그 마지막 부분:\n{log_content[-2000:]}" # Display last 2000 chars of log
        return None, error_message

    # ③ 생성이 끝날 때까지 주기적으로 결과 확인 (폴링)
    t0 = time.time()
    for _ in range(300): # Increased timeout for generation
        time.sleep(2)
        try:
            h = requests.get(f"{SERVER}/history/{pid}", timeout=30).json()
        except Exception:
            continue
        if pid in h:
            entry = h[pid]
            if entry.get("status", {}).get("status_str") == "error":           # ComfyUI 내부 에러
                return None, "ComfyUI 실행 에러: " + json.dumps(entry["status"].get("messages", []), ensure_ascii=False)[:1000]
            outs = entry.get("outputs", {})
            if output_node not in outs or not outs[output_node].get("images"):
                return None, "결과 이미지 없음"
            img = outs[output_node]["images"][0]
            break
    else:
        return None, "타임아웃 (10분 초과)"

    # ④ 완성된 이미지를 다운로드해 Gradio 에 표시
    params = urllib.parse.urlencode({"filename": img["filename"], "subfolder": img.get("subfolder",""), "type": img.get("type","output")})
    rr = requests.get(f"{SERVER}/view?{params}", timeout=30)
    if rr.status_code != 200 or len(rr.content) < 1000:
        return None, f"이미지 다운로드 실패 (status={rr.status_code})"

    local = f"/content/gen_out_{int(time.time())}.png"
    open(local, "wb").write(rr.content)
    arr = np.array(PILImage.open(local).convert("RGB"))   # numpy 배열로 변환
    return arr, f"완료 ({round(time.time()-t0,1)}초) — {int(width)}×{int(height)}, {int(steps)}스텝, 모드: {mode}"


# ---------- Gradio 화면 구성 ----------
with gr.Blocks() as demo:
    gr.Markdown("# FLUX.2-klein-4B / SD1.5 ControlNet 이미지 생성기\n프롬프트를 입력하고 버튼을 누르세요. (4번 셀의 서버가 실행 중이어야 합니다)\n\n※ 포즈 ControlNet 모드는 SD1.5 기반 저용량 파이프라인(~4GB)으로 전환되어 12GB RAM 한계에서도 안전합니다.")

    with gr.Row():
        with gr.Column(scale=3):
            prompt   = gr.Textbox(label="프롬프트", value="A cow flying in the sky with wings on its back, photorealistic", lines=2)
            negative = gr.Textbox(label="네거티브 프롬프트 (선택)", value="", lines=1)
            mode = gr.Radio(["일반 생성 (FLUX.2-klein)", MODE_POSE], value="일반 생성 (FLUX.2-klein)", label="생성 모드")
            pose_image = gr.Image(label="포즈 참고 이미지 (사진 업로드 가능, 자동으로 뼈대 추출)", type="numpy", interactive=True, visible=False)
            controlnet_strength = gr.Slider(0.0, 2.0, value=1.0, step=0.05, label="ControlNet 강도", visible=False)
            btn      = gr.Button("이미지 생성", variant="primary")
            status   = gr.Markdown("")
        with gr.Column(scale=2):
            width  = gr.Slider(512, 1536, value=1024, step=64, label="가로 해상도")
            height = gr.Slider(512, 1536, value=1024, step=64, label="세로 해상도")
            steps  = gr.Slider(1, 40, value=4, step=1, label="스텝 수 (FLUX: 4 권장 / SD1.5 ControlNet: 20~25 권장)")
            seed   = gr.Number(value=42, label="시드(Seed)")

    def toggle_mode(m):
        is_pose = (m == MODE_POSE)
        return (gr.update(visible=is_pose), gr.update(visible=is_pose),
                gr.update(value=25 if is_pose else 4),
                gr.update(value=512 if is_pose else 1024),
                gr.update(value=768 if is_pose else 1024))

    mode.change(toggle_mode, mode, [pose_image, controlnet_strength, steps, width, height])

    out = gr.Image(label="생성 결과")
    btn.click(generate, [prompt, negative, width, height, steps, seed, mode, pose_image, controlnet_strength], [out, status])

# 외부에서 접속 가능한 임시 공개 링크 생성
demo.launch(share=True, quiet=True)

* Running on public URL: https://1cac12ac145c23806f.gradio.live
